<a href="https://colab.research.google.com/github/Sa011c/CITIZEN_ASSIST_AI_SOLU-ES_EM_ENERGIAS_RENOV-VEIS_e_SUSTENT-VEIS/blob/main/citizen_assist_energia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalação e configuração do Ollama no Google Colab
!sudo apt-get update -qq && sudo apt-get install -y zstd pciutils -qq
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama

import subprocess
import time

# Inicia o servidor Ollama em segundo plano
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Baixa o modelo Llama 3.2
!ollama pull llama3.2
print("\n Ollama e modelo Llama 3.2 prontos para uso no Colab!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 4.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package pci.ids.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../pci.ids_0.0~2024.03.31-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2024.03.31-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../libpci3_1%3a3.10.0-2build1_amd64.deb

In [ ]:
# Citizen Assist AI (Solar vs. Biomassa Multiespécie + Área Plantada) + Ollama
import ollama

# ==============================================================================
# CITIZEN ASSIST AI - GS segundo semestre 2026
# Público atendido: Pequenos Produtores Rurais
# Opções de geração de energia: Energia Solar Fotovoltaica vs. Biomassa (Biogás)
# ==============================================================================

MODELO_OLLAMA = "llama3.2"


def calcular_alternativas_rurais(
    consumo_mensal_kwh: float,
    tarifa_kwh: float,
    area_plantada_m2: float,
    qtd_gado: int,
    qtd_aves: int,
    qtd_cabras: int
) -> dict:
    """
    Calcula custos, economia, tempo estimado para retorno do valor investido e
    recomendações comparando Energia Solar e Biomassa (Biogás),
    considerando área plantada e produção de dejetos por espécie.
    """
    custo_mensal_atual = consumo_mensal_kwh * tarifa_kwh
    custo_anual_atual = custo_mensal_atual * 12

    # --- 1ª opção : ENERGIA SOLAR FOTOVOLTAICA ---
    # Premissa: 1 kWp gera em média 120 kWh/mês (CRESESB/CEPEL: 4,5 a 5,0 HSP)
    potencia_solar_kwp = round(consumo_mensal_kwh / 120, 2)
    qtd_placas_550w = max(1, round((potencia_solar_kwp * 1000) / 550))
    # Custo médio da instalação: R$ 3.800,00 por kWp (Greener)
    investimento_solar = potencia_solar_kwp * 3800.00
    # Economia estimada de 88% (descontando taxa mínima / Lei 14.300 - ANEEL)
    economia_anual_solar = custo_anual_atual * 0.88
    payback_solar_anos = round(investimento_solar / economia_anual_solar, 1) if economia_anual_solar > 0 else 0

    # --- 2ª opção: BIOMASSA (BIOGÁS POR ESPÉCIE ANIMAL + ÁREA PLANTADA) ---
    # Premissas técnicas médias de geração elétrica mensal por animal (Embrapa):
    # - 1 Bovino (gado): 25,0 kWh/mês (12 kg dejetos/dia)
    # - 1 Caprino (cabra): 3,0 kWh/mês (1,5 kg dejetos/dia)
    # - 1 Ave (galinha/pato): 0,45 kWh/mês (0,15 kg esterco/dia)
    kwh_gado = qtd_gado * 25.0
    kwh_cabras = qtd_cabras * 3.0
    kwh_aves = qtd_aves * 0.45

    geracao_potencial_biogas_kwh = round(kwh_gado + kwh_cabras + kwh_aves, 1)
    atendimento_biogas_percentual = min(100.0, round((geracao_potencial_biogas_kwh / consumo_mensal_kwh) * 100, 1))
    total_animais = qtd_gado + qtd_cabras + qtd_aves

    # Investimento base de biodigestor rural + filtro H2S + motogerador adaptado (CIBiogás)
    if total_animais > 0:
        investimento_biogas = 18000.00 + (qtd_gado * 150.0) + (qtd_cabras * 25.0) + (qtd_aves * 4.0)
    else:
        investimento_biogas = 0.0

    # Economia proporcional à capacidade de geração (limitada a 90% da conta)
    fator_economia_biogas = min(0.90, (geracao_potencial_biogas_kwh / consumo_mensal_kwh))
    economia_anual_biogas = custo_anual_atual * fator_economia_biogas
    payback_biogas_anos = round(investimento_biogas / economia_anual_biogas, 1) if economia_anual_biogas > 0 else 0

    # Avaliação da área plantada (uso de biofertilizante e restos de cultura)
    area_hectares = area_plantada_m2 / 10000
    if area_plantada_m2 >= 5000:
        nota_lavoura = (
            f"Sua área plantada de {area_plantada_m2:,.0f} m² ({area_hectares:.2f} ha) aproveita "
            "100% do biofertilizante líquido do biodigestor, reduzindo gastos com adubo químico."
        )
    elif area_plantada_m2 > 0:
        nota_lavoura = (
            f"Sua área plantada de {area_plantada_m2:,.0f} m² absorve bem o biofertilizante "
            "gerado como adubo orgânico para hortaliças ou pomar."
        )
    else:
        nota_lavoura = (
            "Como não há área plantada informada, o biofertilizante gerado pelo biodigestor "
            "pode ser vendido ou doado a propriedades vizinhas."
        )

    # --- RECOMENDAÇÃO DO ASSISTENTE ---
    if total_animais > 0 and atendimento_biogas_percentual >= 70:
        recomendacao = (
            "BIOMASSA (BIOGÁS) COMO PRIORIDADE: A combinação dos seus animais gera energia "
            f"suficiente para suprir cerca de {atendimento_biogas_percentual}% do seu consumo mensal. "
            f"Além de cortar a conta de luz e evitar a emissão de gás metano, {nota_lavoura.lower()}"
        )
    elif total_animais > 0 and atendimento_biogas_percentual >= 30:
        recomendacao = (
            "SISTEMA HÍBRIDO (SOLAR + BIOGÁS) OU SOLAR PRINCIPAL: Seus animais suprem "
            f"cerca de {atendimento_biogas_percentual}% da demanda energética. Recomenda-se priorizar "
            "a Energia Solar para cobrir o consumo principal e usar um biodigestor menor para "
            f"tratar os dejetos. {nota_lavoura}"
        )
    elif total_animais > 0:
        recomendacao = (
            "ENERGIA SOLAR FOTOVOLTAICA: Embora você possua criação, o volume atual de animais "
            f"supre apenas ~{atendimento_biogas_percentual}% da sua demanda energética. "
            "A energia solar oferece retorno financeiro mais rápido e exige menos manutenção diária."
        )
    else:
        recomendacao = (
            "ENERGIA SOLAR FOTOVOLTAICA: Como sua propriedade não possui criação animal para geração "
            "contínua de dejetos, a biomassa torna-se inviável. O sistema solar é a melhor opção "
            "pelo menor custo inicial e baixíssima manutenção."
        )

    return {
        "consumo": consumo_mensal_kwh,
        "tarifa": tarifa_kwh,
        "area_plantada_m2": area_plantada_m2,
        "qtd_gado": qtd_gado,
        "qtd_aves": qtd_aves,
        "qtd_cabras": qtd_cabras,
        "custo_mensal_atual": custo_mensal_atual,
        "custo_anual_atual": custo_anual_atual,
        "solar": {
            "potencia_kwp": potencia_solar_kwp,
            "qtd_placas": qtd_placas_550w,
            "investimento": investimento_solar,
            "economia_anual": economia_anual_solar,
            "payback_anos": payback_solar_anos
        },
        "biogas": {
            "kwh_gado": kwh_gado,
            "kwh_aves": kwh_aves,
            "kwh_cabras": kwh_cabras,
            "geracao_estimada_kwh": geracao_potencial_biogas_kwh,
            "atendimento_percentual": atendimento_biogas_percentual,
            "investimento": investimento_biogas,
            "economia_anual": economia_anual_biogas,
            "payback_anos": payback_biogas_anos
        },
        "recomendacao": recomendacao
    }


def criar_contexto_ia(resultado: dict) -> str:
    """Injeta os cálculos exatos na IA Ollama para que ela explique e tire dúvidas sem inventar números."""
    s = resultado["solar"]
    b = resultado["biogas"]
    return f"""
Você é o Citizen Assist AI, um consultor especialista em energias renováveis para pequenos produtores rurais no Brasil.
Use uma linguagem simples, acolhedora e acessível (sem jargões difíceis), fundamentada estritamente nos dados abaixo:

DADOS DA PROPRIEDADE:
- Consumo: {resultado['consumo']} kWh/mês | Tarifa: R$ {resultado['tarifa']:.2f}/kWh
- Gasto atual: R$ {resultado['custo_mensal_atual']:.2f}/mês (R$ {resultado['custo_anual_atual']:.2f}/ano)
- Área plantada: {resultado['area_plantada_m2']} m²
- Animais informados: {resultado['qtd_gado']} bovinos, {resultado['qtd_aves']} aves, {resultado['qtd_cabras']} cabras.

ALTERNATIVA 1 - ENERGIA SOLAR:
- Sistema: {s['potencia_kwp']} kWp ({s['qtd_placas']} placas de 550W)
- Investimento: R$ {s['investimento']:,.2f} | Economia: R$ {s['economia_anual']:,.2f}/ano | Payback: {s['payback_anos']} anos

ALTERNATIVA 2 - ENERGIA DE BIOMASSA (BIOGÁS):
- Geração potencial: {b['geracao_estimada_kwh']} kWh/mês ({b['atendimento_percentual']}% do consumo)
- Investimento: R$ {b['investimento']:,.2f} | Economia: R$ {b['economia_anual']:,.2f}/ano | Payback: {b['payback_anos']} anos

RECOMENDAÇÃO TÉCNICA DO SISTEMA: {resultado['recomendacao']}

REGRAS OBRIGATÓRIAS:
1. Explique em um parágrafo curto e didático por que a recomendação acima faz sentido para o produtor, considerando os animais e a área plantada.
2. Sempre reforce que os valores são estimativas baseadas em médias (ANEEL, CRESESB/CEPEL, Embrapa e CIBiogás) e não constituem garantia financeira.
3. Responda às dúvidas futuras do produtor mantendo foco em pequenos produtores rurais, energia solar e biomassa.
"""


def perguntar_sim_nao(pergunta: str) -> bool:
    while True:
        resp = input(pergunta).strip().lower()
        if resp in ["s", "sim", "y", "yes"]:
            return True
        elif resp in ["n", "nao", "não", "no"]:
            return False
        print(" Por favor, responda apenas com 'S' (Sim) ou 'N' (Não).")


def executar_assistente():
    print("=" * 75)
    print("🌱 CITIZEN ASSIST AI - CONSULTOR DE ENERGIA PARA O PRODUTOR RURAL 🌱")
    print("=" * 75)
    print("Olá! Vou te ajudar a comparar Energia Solar e Biomassa (Biogás) para a sua propriedade.\n")

    try:
        consumo = float(input("1. Qual o consumo médio mensal da propriedade em kWh? (Ex: 500): "))
        tarifa_input = input("2. Qual o valor da tarifa rural em R$/kWh? (Pressione ENTER para usar R$ 0.65): ")
        tarifa = float(tarifa_input.replace(",", ".")) if tarifa_input.strip() != "" else 0.65
        area_plantada = float(input("3. Qual a metragem do espaço plantado em m²? (Ex: 10000 para 1 hectare, ou 0): "))

        print("\n4. Sobre a criação de animais na propriedade:")
        qtd_gado = 0
        if perguntar_sim_nao("   - Você possui gado (bovinos)? (S/N): "):
            qtd_gado = int(input("     -> Quantas cabeças de gado você possui? "))

        qtd_aves = 0
        if perguntar_sim_nao("   - Você possui aves (galinhas, patos, etc.)? (S/N): "):
            qtd_aves = int(input("     -> Quantas aves você possui? "))

        qtd_cabras = 0
        if perguntar_sim_nao("   - Você possui cabras (caprinos)? (S/N): "):
            qtd_cabras = int(input("     -> Quantas cabras você possui? "))

        if consumo <= 0 or tarifa <= 0 or area_plantada < 0 or qtd_gado < 0 or qtd_aves < 0 or qtd_cabras < 0:
            print("\n Por favor, insira valores válidos (consumo e tarifa maiores que zero).")
            return

        resultado = calcular_alternativas_rurais(
            consumo, tarifa, area_plantada, qtd_gado, qtd_aves, qtd_cabras
        )

        print("\n" + "=" * 75)
        print(" DIAGNÓSTICO ENERGÉTICO DA PROPRIEDADE")
        print("=" * 75)
        print(f"• Consumo analisado: {consumo:.0f} kWh/mês ({consumo*12:.0f} kWh/ano)")
        print(f"• Área plantada informada: {area_plantada:,.0f} m² ({area_plantada/10000:.2f} hectares)")
        print(f"• Gasto atual estimado na rede elétrica: R$ {resultado['custo_mensal_atual']:.2f}/mês (R$ {resultado['custo_anual_atual']:.2f}/ano)\n")

        print("--- ALTERNATIVA 1: ENERGIA SOLAR FOTOVOLTAICA ---")
        s = resultado["solar"]
        print(f"• Tamanho do sistema: {s['potencia_kwp']} kWp (aprox. {s['qtd_placas']} placas de 550W)")
        print(f"• Investimento estimado: R$ {s['investimento']:,.2f}")
        print(f"• Economia anual estimada: R$ {s['economia_anual']:,.2f}/ano")
        print(f"• Tempo de retorno (Payback): {s['payback_anos']} anos")
        print("• Impacto ambiental: Zero emissão na operação; usa área de telhado existente.\n")

        print("--- ALTERNATIVA 2: ENERGIA DE BIOMASSA (BIOGÁS) ---")
        b = resultado["biogas"]
        total_animais = qtd_gado + qtd_aves + qtd_cabras
        if total_animais > 0:
            print("• Detalhamento da geração estimada por espécie:")
            if qtd_gado > 0:
                print(f"   - Gado ({qtd_gado} un.): {b['kwh_gado']:.1f} kWh/mês")
            if qtd_aves > 0:
                print(f"   - Aves ({qtd_aves} un.): {b['kwh_aves']:.1f} kWh/mês")
            if qtd_cabras > 0:
                print(f"   - Cabras ({qtd_cabras} un.): {b['kwh_cabras']:.1f} kWh/mês")
            print(f"• Potencial total de geração: {b['geracao_estimada_kwh']:.1f} kWh/mês ({b['atendimento_percentual']}% do consumo)")
            print(f"• Investimento estimado (Biodigestor + Gerador): R$ {b['investimento']:,.2f}")
            print(f"• Economia anual estimada: R$ {b['economia_anual']:,.2f}/ano")
            print(f"• Tempo de retorno (Payback): {b['payback_anos']} anos")
            print("• Impacto ambiental: Evita emissão de metano (CH4) e gera biofertilizante líquido.")
        else:
            print("• Inviável no cenário atual: Nenhum animal informado para produção contínua de biogás.")

        print("\n" + "-" * 75)
        print(" RECOMENDAÇÃO DO CITIZEN ASSIST AI:")
        print(resultado["recomendacao"])
        print("-" * 75)

        print("\n CONDIÇÕES E LIMITAÇÕES DAS ESTIMATIVAS:")
        print("1. Premissas de biogás por espécie (Embrapa): 25 kWh/mês por bovino, 3 kWh/mês por caprino e 0,45 kWh/mês por ave.")
        print("2. Os cálculos solares utilizam tarifas médias rurais (ANEEL) e irradiação média brasileira (CRESESB).")
        print("3. No biogás, a eficiência depende do confinamento dos animais (coleta de dejetos) e temperatura.")
        print("4. Estas estimativas servem como apoio à decisão e não constituem garantia financeira.")

        print("\n FONTES E REFERÊNCIAS UTILIZADAS:")
        print("• ANEEL (Agência Nacional de Energia Elétrica) - Tarifas de Energia Rural.")
        print("• CRESESB/CEPEL - Potencial Solar SunData.")
        print("• EPE (Empresa de Pesquisa Energética) - Anuário Estatístico de Energia Elétrica.")
        print("• Embrapa / CIBiogás - Cartilhas de Biogás e Biodigestores Rurais.")
        print("=" * 75)

        # --- INTEGRAÇÃO COM A OLLAMA ---
        print("\n ORIENTAÇÃO PERSONALIZADA DA IA (OLLAMA):")
        mensagens = [
            {"role": "system", "content": criar_contexto_ia(resultado)},
            {"role": "user", "content": "Explique esse resultado em linguagem simples para mim, como um pequeno produtor rural."}
        ]

        resposta_ia = ""
        for chunk in ollama.chat(model=MODELO_OLLAMA, messages=mensagens, stream=True):
            texto = chunk["message"]["content"]
            print(texto, end="", flush=True)
            resposta_ia += texto

        mensagens.append({"role": "assistant", "content": resposta_ia})
        print("\n" + "=" * 75)

        # Chat interativo contínuo para retirar dúvidas
        print("\n Tem alguma dúvida sobre energia solar ou biogás na sua propriedade? Pergunte abaixo (ou digite 'sair'):")
        while True:
            pergunta = input("\n Produtor: ")
            if pergunta.strip().lower() in ["sair", "exit", "quit"]:
                print(" Obrigado por usar o Citizen Assist AI! Até a próxima.")
                break

            mensagens.append({"role": "user", "content": pergunta})
            print(" Assistente: ", end="")
            resposta_turno = ""
            for chunk in ollama.chat(model=MODELO_OLLAMA, messages=mensagens, stream=True):
                texto = chunk["message"]["content"]
                print(texto, end="", flush=True)
                resposta_turno += texto
            print()
            mensagens.append({"role": "assistant", "content": resposta_turno})

    except ValueError:
        print("\n Erro: Digite apenas números válidos (use ponto para decimais, ex: 0.65).")
    except Exception as e:
        print(f"\n Erro de conexão com o Ollama: {e}")
        print("Verifique se você executou a Célula 1 antes de rodar esta célula.")


if __name__ == "__main__":
    executar_assistente()

🌱 CITIZEN ASSIST AI - CONSULTOR DE ENERGIA PARA O PRODUTOR RURAL 🌱
Olá! Vou te ajudar a comparar Energia Solar e Biomassa (Biogás) para a sua propriedade.

1. Qual o consumo médio mensal da propriedade em kWh? (Ex: 500): 313
2. Qual o valor da tarifa rural em R$/kWh? (Pressione ENTER para usar R$ 0.65): 0.72
3. Qual a metragem do espaço plantado em m²? (Ex: 10000 para 1 hectare, ou 0): 0

4. Sobre a criação de animais na propriedade:
   - Você possui gado (bovinos)? (S/N): 0
 Por favor, responda apenas com 'S' (Sim) ou 'N' (Não).
   - Você possui gado (bovinos)? (S/N): n
   - Você possui aves (galinhas, patos, etc.)? (S/N): s
     -> Quantas aves você possui? 9
   - Você possui cabras (caprinos)? (S/N): s
     -> Quantas cabras você possui? 4

 DIAGNÓSTICO ENERGÉTICO DA PROPRIEDADE
• Consumo analisado: 313 kWh/mês (3756 kWh/ano)
• Área plantada informada: 0 m² (0.00 hectares)
• Gasto atual estimado na rede elétrica: R$ 225.36/mês (R$ 2704.32/ano)

--- ALTERNATIVA 1: ENERGIA SOLAR FOTOV